# Silver Layer - Products Enriched Table

## Purpose
Transform raw Bronze products into enriched, clean Silver products table with category and pricing data.

## Input
* **Source:** `big_data.bronze.products`
* **Enrichment:** `big_data.bronze.aisles`, `big_data.bronze.departments`, `big_data.bronze.prices`
* **Rows:** ~49.7K

## Output
* **Target:** `big_data.silver.products_enriched`
* **Rows:** ~49.7K
* **Primary Key:** `product_id`

## Transformations

### Step 1: Load, Type Cast and Category Enrichment
* Cast numeric columns to proper types
* LEFT JOIN with aisles (get aisle name)
* LEFT JOIN with departments (get department name)
* String normalization (trim, remove quotes)
* Filter NULL primary keys

### Step 2: Price Enrichment and Feature Engineering
* LEFT JOIN with prices (fuzzy match on normalized product_name)
* Cast price_usd to decimal(10,2)
* Price validation: filter invalid prices (<=0)
* **Derive price_band**: Very Low / Low / Medium / High / Premium / Luxury
* Add silver_timestamp for tracking

## Data Quality Validations
* NOT NULL on primary key and critical columns
* UNIQUE constraint on product_id
* Enrichment coverage check (aisle, department, price)
* Price validation (>0 or NULL)

## Persistence
Only persists to Delta table if **all validations pass**.

## Execution
Run all cells sequentially. Expected runtime: ~2-3 minutes.

In [0]:
%run ../UTILS/data_quality_checks

In [0]:
# PySpark imports
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DecimalType

In [0]:
# Schema configuration
source_schema = "big_data.bronze"
target_schema = "big_data.silver"

# Table names
source_table = "products"
target_table = "products_enriched"

# Enrichment tables
aisles_table = "aisles"
departments_table = "departments"
prices_table = "prices"

# Print configuration
print("Configuration:")
print(f"  Source: {source_schema}.{source_table}")
print(f"  Target: {target_schema}.{target_table}")
print(f"  Enrichment: {aisles_table}, {departments_table}, {prices_table}")

In [0]:
print("Step 1: Loading, casting types and enriching with categories...")

# Load products and apply type casting
products_df = spark.table(f"{source_schema}.{source_table}") \
    .withColumn("product_id",    F.col("product_id").cast(IntegerType())) \
    .withColumn("aisle_id",      F.col("aisle_id").cast(IntegerType())) \
    .withColumn("department_id", F.col("department_id").cast(IntegerType())) \
    .withColumn("product_name",  F.trim(F.col("product_name"))) \
    .filter(F.col("product_id").isNotNull()) \
    .filter(F.col("product_name").isNotNull())

# Join with aisles
aisles_df = spark.table(f"{source_schema}.{aisles_table}").select("aisle_id", "aisle")
products_df = products_df.join(aisles_df, on="aisle_id", how="left")

# Join with departments
departments_df = spark.table(f"{source_schema}.{departments_table}").select("department_id", "department")
products_df = products_df.join(departments_df, on="department_id", how="left")

print(f"  Loaded and enriched: {products_df.count():,} rows")
print(f"  Joined with aisles and departments")

In [0]:
print("Step 2: Enriching with prices and deriving features...")

# Normalize product names for fuzzy matching
products_df = products_df \
    .withColumn("product_name_normalized", 
        F.trim(F.regexp_replace(F.col("product_name"), "^'+|'+$", ""))
    )

# Load and prepare prices
prices_df = spark.table(f"{source_schema}.{prices_table}").select(
    F.trim(F.regexp_replace(F.col("product_name"), "^'+|'+$", "")).alias("price_product_name"),
    F.expr("try_cast(price_usd as decimal(10,2))").alias("price_usd")
)

# Join with prices (fuzzy match on normalized names)
products_silver = products_df \
    .join(
        prices_df,
        on=F.col("product_name_normalized") == F.col("price_product_name"),
        how="left"
    ) \
    .drop("price_product_name", "product_name_normalized") \
    .filter((F.col("price_usd").isNull()) | (F.col("price_usd") > 0))

# Derive price_band
products_silver = products_silver \
    .withColumn("price_band",
        F.when(F.col("price_usd").isNull(), None)
         .when(F.col("price_usd") < 2.50, "Very Low")
         .when(F.col("price_usd") < 5.00, "Low")
         .when(F.col("price_usd") < 9.00, "Medium")
         .when(F.col("price_usd") < 15.00, "High")
         .when(F.col("price_usd") < 30.00, "Premium")
         .otherwise("Luxury")
    ) \
    .withColumn("_silver_timestamp", F.current_timestamp()) \
    .drop("ingestion_timestamp", "source_file")

print(f"  Transformed: {products_silver.count():,} rows")
print(f"  Derived feature: price_band")

print("\nPreview:")
products_silver.select("product_id", "product_name", "aisle", "department", "price_usd", "price_band").show(5, truncate=False)

In [0]:
print_validation_header("products_enriched - Technical Validations")

total_rows = products_silver.count()
print(f"\nTotal rows: {total_rows:,}")
print(f"Expected: ~49.7K rows\n")

# Initialize validation flag
validation_passed_technical = True

# 1. NOT NULL checks (Primary Key)
status, failed, msg = check_not_null(products_silver, ["product_id"])
print_check_result("NOT NULL (PK)", status, msg, failed)
if status == "FAIL":
    validation_passed_technical = False

# 2. UNIQUE check (Primary Key)
status, duplicates, msg = check_unique(products_silver, ["product_id"])
print_check_result("UNIQUE (product_id)", status, msg, duplicates)
if status == "FAIL":
    validation_passed_technical = False

# 3. NOT NULL checks (Critical columns)
status, failed, msg = check_not_null(products_silver, ["product_name", "aisle_id", "department_id"])
print_check_result("NOT NULL (Critical columns)", status, msg, failed)
if status == "FAIL":
    validation_passed_technical = False

print("\n" + "="*60)
if validation_passed_technical:
    print("SUCCESS: Technical validations PASSED")
else:
    print("FAILURE: Technical validations FAILED")
print("="*60)

In [0]:
print_validation_header("products_enriched - Business Validations")

# Initialize business validation flag
validation_passed_business = True

# 1. Enrichment coverage
print("\n1. Enrichment Coverage:")
with_aisle = products_silver.filter(F.col("aisle").isNotNull()).count()
with_dept = products_silver.filter(F.col("department").isNotNull()).count()
with_price = products_silver.filter(F.col("price_usd").isNotNull()).count()

print(f"  Aisle: {with_aisle:,}/{total_rows:,} ({with_aisle/total_rows*100:.1f}%)")
print(f"  Department: {with_dept:,}/{total_rows:,} ({with_dept/total_rows*100:.1f}%)")
print(f"  Price: {with_price:,}/{total_rows:,} ({with_price/total_rows*100:.1f}%)")

if with_aisle < total_rows * 0.95 or with_dept < total_rows * 0.95:
    print("  ⚠️ WARNING: Less than 95% enrichment coverage")
    validation_passed_business = False

# 2. Price validation (no invalid prices)
print("\n2. Price Validation:")
invalid = products_silver.filter((F.col("price_usd").isNotNull()) & (F.col("price_usd") <= 0)).count()
status = "PASS" if invalid == 0 else "FAIL"
print_check_result("PRICE VALIDATION (>0)", status, "All prices valid" if invalid == 0 else f"{invalid:,} invalid prices", invalid)
if status == "FAIL":
    validation_passed_business = False

# 3. Price band distribution
print("\n3. Price Band Distribution:")
products_silver.groupBy("price_band").count().orderBy("price_band").show()

print("\n" + "="*60)
if validation_passed_business:
    print("SUCCESS: Business validations PASSED")
else:
    print("FAILURE: Business validations FAILED")
print("="*60)

In [0]:
# Combine technical and business validation results
validation_passed = validation_passed_technical and validation_passed_business

print("\n" + "="*60)
print("OVERALL VALIDATION RESULT")
print("="*60)
print(f"  Technical Validation: {'PASSED ✓' if validation_passed_technical else 'FAILED ✗'}")
print(f"  Business Validation:  {'PASSED ✓' if validation_passed_business else 'FAILED ✗'}")
print("="*60)

if validation_passed:
    print(f"\n✓ ALL VALIDATIONS PASSED - Ready to persist to Silver layer")
else:
    print(f"\n✗ SOME VALIDATIONS FAILED - Will NOT persist")
    print("\nPlease review and fix the errors above before re-running.")

print("="*60)

In [0]:
# Only persist if validation passed
if validation_passed:
    print("Persisting to Silver layer...")
    
    products_silver.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{target_schema}.{target_table}")
    
    # Verify
    final_count = spark.table(f"{target_schema}.{target_table}").count()
    
    print("\n" + "="*60)
    print("SUCCESS: Products Enriched table persisted to Silver layer")
    print("="*60)
    print(f"\nFinal Statistics:")
    print(f"  Table: {target_schema}.{target_table}")
    print(f"  Rows: {final_count:,}")
    print(f"  Primary Key: product_id")
    print(f"  Format: Delta")
    print("\nNext Step: Run silver_order_products notebook")
else:
    print("\n" + "="*60)
    print("ABORTED: Validation failed - table NOT persisted")
    print("="*60)
    print("\nFix validation errors above and re-run.")